In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "46d487bb9b699b97c7c5fc5edd866e4d2ad7919b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/46d487bb9b699b97c7c5fc5edd866e4d2ad7919b/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Hardware

The previous section characterized a GPU by two constants—its peak arithmetic
rate $P$ and its memory bandwidth $\beta$—and showed that an operation
performing $I$ FLOPs per byte runs at most at $\min(P, I\beta)$. On the
RTX 4090, $P = 165$ TFLOP/s for bf16 matrix products and $\beta \approx 1$
TB/s: in the time the card reads one byte from memory, it can execute 165
floating-point operations. Where do these two numbers come from, and why does
moving a byte take so much longer than a FLOP?

Consider generating text one token at a time with a 1B-parameter GPT
(that section). In bf16 its weights occupy 2 GB, and every generated
token reads all of them. The arithmetic for one token ($2 \times 10^9$ FLOPs)
takes the tensor cores 12 µs, while reading 2 GB of weights at the achieved
0.9 TB/s takes 2.2 ms. A single stream therefore decodes at most ~450
tokens/s, leaving the arithmetic units idle over 99% of the time.

Three physical facts explain this gap: data must cross boundaries between
chips; off-chip wires scale more slowly than on-chip transistors; and moving
a byte consumes orders of magnitude more energy than computing with it.
Together they explain why the ridge point $P/\beta$ is in the hundreds and has
risen over hardware generations. Our worked example is the workstation that
runs this book's notebooks (four RTX 4090 GPUs, late 2026);
@Hennessy.Patterson.2011 treat computer architecture in depth, and
that section discusses hardware selection.

In [1]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## The Memory Hierarchy

Every byte a program touches resides in a hierarchy of memories that trade
capacity for speed (the figure). On-die registers and
SRAM caches hold ~0.1 GB and deliver tens of TB/s. The GPU's off-die DRAM—*high-bandwidth
memory* (HBM) on data-center cards or GDDR on consumer cards—holds tens to
hundreds of GB at 1–8 TB/s. Host DRAM holds hundreds of GB at ~0.5 TB/s per
CPU socket, NVMe flash holds terabytes at ~14 GB/s, and network storage sits
further out still.

![The memory hierarchy in late 2026. Each level downward holds orders of
magnitude more bytes and delivers orders of magnitude fewer of them per
second.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-memory-hierarchy.svg)

### Bandwidth and Latency

the figure ranks the data paths of a training job across
nearly three orders of magnitude, from 14 GB/s (NVMe) to 8 TB/s (NVIDIA B200
HBM). One rule explains every step: **bandwidth equals the number of data
wires times the bit rate per wire**. The RTX 4090 accesses its GDDR6X memory
over 384 wires at 21 Gb/s each (~1 TB/s), whereas its PCIe 4.0 ×16 slot has
16 lanes per direction at 16 Gb/s each (32 GB/s)—roughly the same speed per
wire, but $24\times$ fewer wires.

Distance sets the wire count. HBM stacks sit inside the GPU package
millimeters from the die, connected by thousands of short wires; board traces
support hundreds of wires, and slots or cables a few dozen lanes. **Each step
away from the die cuts the wire count, and with it the bandwidth, by roughly an
order of magnitude.** A B200 reads HBM at 8 TB/s, reaches peer GPUs over
NVLink 5 at 0.9 TB/s (using ~36 lanes at 200 Gb/s each), and reaches the host
over PCIe 5.0 at 64 GB/s.

![Bandwidth of the data paths that a training job uses, late 2026. Links
count one direction. The memory and host link of our machine are shown in
orange.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-bandwidth-ladder.svg)

Bandwidth governs steady-state flow; *latency* is the time for the first byte
to arrive (the figure, with RTX 4090 microbenchmarks from
@Luo.Fan.Li.ea.2024). One eager operation dispatched from Python takes
10–100 µs of host time (of which the hardware kernel launch is 1–5 µs). In
10 µs the RTX 4090 can stream 10 MB, which is why small operations are
dominated by dispatch overhead rather than memory or compute.

![Latencies on a logarithmic scale. The highlighted entry is the host time
of one framework operation dispatched from Python, of which the kernel
launch is a part.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-latency-ladder.svg)

Latency and bandwidth interact through the *access pattern*. DRAM achieves its
rated bandwidth only on sequential addresses. Let us gather a 1 GiB fp32
vector `x[idx]` once with `idx` equal to the identity permutation and once
with a random permutation. GPU memory transfers data in 32-byte *sectors*, so
each random 4-byte read fetches an entire 32-byte sector while the index and
output vectors still stream sequentially (~$3\times$ more bytes moved in all).

In [2]:
n = 256 * 1024 * 1024
x = torch.randn(n, device=d2l.try_gpu())  # 1 GiB fp32
times = {}
for desc in ('identity', 'random'):
    idx = (torch.arange(n, device=x.device) if desc == 'identity' else
           torch.randperm(n, device=x.device))
    # Bytes per element: the index and the output stream; x is read as
    # 4 bytes in order, or as a whole 32-byte sector at a random address
    nbytes = idx.dtype.itemsize + 4 + (4 if desc == 'identity' else 32)
    times[desc] = d2l.Benchmark(lambda: x[idx]).time
    print(f'{desc:8s}: {1e3 * times[desc]:5.1f} ms, '
          f'{nbytes * n / times[desc] / 1e12:.2f} TB/s at {nbytes} B/element')
    del idx  # Free the 2 GiB index before making the next one
print(f'random / identity: {times["random"] / times["identity"]:.1f}x')

identity:   4.8 ms, 0.90 TB/s at 16 B/element


random  :  37.4 ms, 0.32 TB/s at 44 B/element
random / identity: 7.8x


The identity gather streams at ~0.9 TB/s, whereas the random gather takes
~37 ms ($8\text{--}11\times$ slower) and achieves only a third of peak
bandwidth even when credited with a full 32-byte sector per read. Beyond
32-byte sectors, DRAM serves data from open *rows* of a few kilobytes;
consecutive addresses share an open row, whereas random addresses repeatedly
pay the latency of opening a new row (and miss address-translation caches).

Dense matrix products and embedding lookups (which read contiguous rows of
thousands of bytes per token) stream efficiently. In contrast, fine-grained
random accesses (sparse matrices, graph traversals, hash tables) run at a
fraction of $\beta$. More generally, transferring a contiguous block of $n$
bytes with start-up latency $\alpha$ takes $t(n) = \alpha + n/\beta$ and
reaches half of peak bandwidth at $n_{1/2} = \alpha\beta$ [@Hockney.1994].
For an NVMe drive ($\alpha \approx 50\text{--}100$ µs, $\beta = 14$ GB/s),
$n_{1/2} \approx 1$ MB, which is why datasets on disk are shuffled in
megabyte-scale shards rather than individual examples.

## Compute and Memory Bandwidth

### Area and Perimeter

Arithmetic units fill the interior of a silicon die (scaling with area,
$O(n^2)$), whereas I/O pads sit along the perimeter, or *shoreline* (scaling
with edge length, $O(n)$, the figure). Shrinking transistors has
the same effect: logic density quadruples when feature size halves, while
solder-bump pitch at the die edge shrinks much more slowly.

![Compute is area, I/O is perimeter. Doubling the side of a die quadruples
its compute but only doubles the room for
I/O.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-shoreline.svg)

Architects mitigate the shoreline bottleneck in two ways:

1. **HBM on a silicon interposer:** Stacking DRAM dies vertically next to the
   GPU die replaces a few hundred board traces with thousands of short, dense
   interposer wires (giving a B200 8 TB/s versus ~0.5 TB/s for a CPU socket).
2. **Multi-die packages:** Because lithography caps single-die area at the
   *reticle limit* (~$858\,\textrm{mm}^2$), flagship accelerators such as the
   B200 join two reticle-limit dies on one package, doubling the perimeter to
   accommodate eight HBM stacks.

### Generational Trends

Across NVIDIA's data-center generations from V100 (2017) through A100 and H100
to B200 (2024), dense 16-bit FLOP/s grew by ~$2.6\times$ per generation, memory
bandwidth by ~$2.0\times$, and capacity by ~$1.8\times$
[@Gholami.Yao.Kim.ea.2024]. Consequently, the ridge point doubled from
139 FLOP/byte on the V100 to 295 on the H100 (281 on the B200): an operation
with intensity 200 FLOP/byte was compute-bound on a V100 but is
bandwidth-bound on an H100 or B200.

### Energy per Operation

Power sets a second hard cap on bandwidth. In a 45 nm CMOS process
(the figure, @Horowitz.2014), an fp32 multiply costs
3.7 pJ and an 8-bit integer add 0.03 pJ, whereas reading 64 bits from off-chip
DRAM costs 1.3–2.6 nJ—roughly the energy of **500 fp32 multiplications**.
On-chip SRAM reads lie in between (10–100 pJ depending on distance). While
modern process nodes scale all numbers down, the hierarchy remains unchanged:
moving data costs far more energy than computing on it. Kernel fusion,
narrower number formats, and activation recomputation save power for the exact
same reason they save time.

![Energy per operation in a 45 nm process. One 64-bit DRAM read costs
about as much as 500 fp32
multiplications.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-energy-ladder.svg)

## The GPU

### Streaming Multiprocessors

A GPU dedicates its die area to thousands of arithmetic units organized into
*streaming multiprocessors* (SMs; 128 on the RTX 4090). Each SM executes
threads in lockstep groups of 32 called *warps*. Instead of relying on massive
caches to hide DRAM latency, an SM switches warps on every cycle so other
warps compute while stalled warps wait for memory.

By Little's law [@Little.1961], keeping a memory pipe of bandwidth
$\beta = 1.0$ TB/s full across a latency of 250 ns requires
$1.0\,\textrm{TB/s} \times 250\,\textrm{ns} \approx 250$ KB in flight at all
times (~2 KB per SM, or 16 warps per SM each waiting on a 4-byte load per
thread). Kernels that expose too little parallelism to keep 250 KB in flight
become latency-bound rather than bandwidth-bound.

### Tensor Cores and Number Formats

*Tensor cores* multiply and accumulate small matrix tiles in a single
instruction, amortizing instruction fetch and decode over an entire tile, much
like a TPU systolic array [@Kung.1988; @Jouppi.Young.Patil.ea.2017]. On an
H100, bf16 tensor cores deliver 989 TFLOP/s versus 67 TFLOP/s for scalar fp32
($15\times$); on the consumer RTX 4090, the ratio is $2\times$.

We can derive the RTX 4090's peak throughput directly from its specification:
128 SMs $\times$ 128 fp32 units $= 16{,}384$ units, each executing one fused
multiply-add (2 FLOPs) per cycle at the 2.52 GHz boost clock:

$$
P_\textrm{fp32} = 16{,}384 \times 2 \times 2.52 \times 10^{9}\ \textrm{s}^{-1}
\approx 82.6\ \textrm{TFLOP/s}.
$$

Its tensor cores run tf32 at the same 82.6 TFLOP/s, bf16 (with fp32
accumulation) at $2\times = 165.2$ TFLOP/s, and fp8 nominally at
$4\times = 330.4$ TFLOP/s. We measure the tf32 and bf16 rates in this
chapter; the fp8 rate is a nominal figure that none of our code exercises.

![Floating-point formats drawn bit for bit. fp32, tf32, and bf16 share an
8-bit exponent and hence a range. tf32 is a compute mode of the tensor
cores whose values are stored in 32 bits; every other step down halves the
bytes per value.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-float-formats.svg)

As the figure illustrates, narrowing the format has two
roofline effects:

1. **Halving the width halves the bytes moved** (speeding up bandwidth-bound
   kernels up to $2\times$ and halving weight storage), except for tf32, which
   computes on 19-bit values (a 10-bit mantissa) inside the tensor cores but
   still stores 32-bit floats in memory.
2. **Halving the width usually doubles the tensor-core compute roof $P$.**

Formats of 8 bits or fewer require explicit scale factors. fp8 uses E4M3 for
forward activations/weights and E5M2 for gradients with per-tensor or per-block
scales [@Micikevicius.Stosic.Burgess.ea.2022]. At 4 bits (fp4 / E2M1,
encoding only 15 values from $-6$ to $6$), per-tensor scaling fails on
outliers, so hardware applies *microscaling*—one shared scale per block of 16
(NVFP4) or 32 (OCP MX) elements [@Rouhani.Zhao.More.ea.2023].

### Accelerator Specifications

the table compares two data-center GPUs and two consumer cards.

<!-- tbl-caption:Four widely deployed GPUs. Dense tensor-core throughput with fp32 accumulation at boost clock, in TFLOP/s (TF). The fp8 and fp4 entries of the two consumer cards are nominal: fp8 is twice the bf16 rate, and the RTX 5090's fp4 entry follows the vendor's headline figure, not a measurement. Links count one direction. -->

| | H100 SXM | B200 | RTX 4090 (ours) | RTX 5090 |
|---|---|---|---|---|
| memory $C$ | 80 GB HBM3 | 180 GB HBM3e | 24 GB GDDR6X | 32 GB GDDR7 |
| memory bandwidth $\beta$ | 3.35 TB/s | 8 TB/s | 1.0 TB/s | 1.8 TB/s |
| bf16 | 989 TF | 2,250 TF | 165 TF | 210 TF |
| fp8 | 1,979 TF | 4,500 TF | 330 TF | 419 TF |
| fp4 | — | 9,000 TF | — | 1,676 TF |
| ridge point (bf16) | 295 FLOP/B | 281 FLOP/B | 164 FLOP/B | 117 FLOP/B |
| link to other GPUs | NVLink 4, 450 GB/s | NVLink 5, 900 GB/s | PCIe 4.0 ×16 via host, $\le 32$ GB/s | PCIe 5.0 ×16 via host, $\le 64$ GB/s |
| power | 700 W | 1,000 W | 450 W | 575 W |

## Data Paths outside the GPU

### The Host

While the GPU performs the arithmetic, the host CPU still performs three tasks
that can bottleneck training: dispatching kernels ($h$ per call), running the
data-loading and preprocessing pipeline, and staging host-to-device transfers
over PCIe.

The GPU's DMA *copy engine* can transfer data over PCIe concurrently with SM
computation only from page-locked (*pinned*) host memory. Transfers from
ordinary *pageable* memory must first be copied by the CPU into an internal
pinned staging buffer. Let us compare pageable and pinned transfer rates for
a 256 MiB array.

In [3]:
x_pageable = torch.randn(64 * 1024 * 1024)  # 256 MiB on the host
x_pinned = x_pageable.pin_memory()
gpu = d2l.try_gpu()
for desc, src in [('pageable', x_pageable), ('pinned', x_pinned)]:
    t = d2l.Benchmark(lambda: src.to(gpu, non_blocking=True)).time
    print(f'{desc}: {src.nbytes / t / 1e9:.1f} GB/s')

pageable: 14.2 GB/s
pinned: 23.1 GB/s


The pinned transfer reaches ~23 GB/s—about three quarters of PCIe 4.0 ×16's
32 GB/s theoretical ceiling, and $40\times$ slower than GPU memory—while the
pageable transfer is slower due to the extra host-memory copy. Datasets that
fit in GPU memory should stay on the device; minibatches streamed from the
host should use pinned host memory overlapped with compute.

A `DataLoader` with `pin_memory=True` stages its minibatches in pinned memory.

### Links between GPUs

Data-center GPUs communicate over dedicated NVLink fabrics (450 GB/s per
direction on H100, 900 GB/s on B200). Consumer RTX 4090 cards lack both NVLink
and PCIe peer-to-peer (P2P) access: as the figure shows,
every inter-GPU byte on our four-GPU workstation must travel up one PCIe 4.0
link, through host DRAM, and down another, capping GPU-to-GPU transfers at the
~23 GB/s pinned-copy rate ($40\times$ slower than NVLink 5).

![Our machine, as reported by `nvidia-smi topo`: four RTX 4090s in pairs
behind two PCIe host bridges, with neither peer-to-peer access nor NVLink.
The highlighted path is the route of every byte from one GPU to
another.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-pcie-topology.svg)

For our 1B-parameter GPT ($S = 4$ GB of fp32 gradients), moving the gradients
once at 23 GB/s takes at least 0.17 s on our workstation versus < 5 ms over
NVLink 5—more than half of the 0.3 s compute time of an 8,192-token step at
peak bf16 throughput.

## Workloads on the Roofline

Language model inference spans two regimes
(that section). *Prefill* processes a prompt of
$n$ tokens in parallel, achieving intensity $I \approx n$ (up to the layer
width) and running on the compute roof. *Decode* generates one token per step,
reading all $2N$ bytes of bf16 weights plus the KV cache for an intensity of
$I \approx 1$ FLOP/byte.

### Single-Stream and Batched Decoding

Because single-stream decoding is bandwidth-bound, its token rate is bounded
by

$$
\textrm{tokens/s} \;\lesssim\;
\frac{\beta}{\textrm{bytes per token}} \;=\;
\frac{\beta}{2N + \textrm{KV bytes}}.
$$

When a model's weights fill a card's memory capacity $C$, the maximum
single-stream decode rate is the inverse of the *memory sweep time* $C/\beta$:
the time required to read every byte of GPU DRAM once. Across all four GPUs in
the table, capacity and bandwidth vary by $7\text{--}8\times$,
yet $C/\beta$ stays near **20 ms** (24 ms on H100 and RTX 4090, 22.5 ms on
B200, 18 ms on RTX 5090). Any dense model that fills its GPU therefore decodes
a single stream at roughly 40–55 tokens/s, whether on a consumer card or a
flagship data-center GPU. The bound assumes that every token reads every
weight. A mixture-of-experts model (that section) reads only the experts
that its router selects for a token, so it decodes faster than $C/\beta$
suggests.

Batching $B$ concurrent sequences amortizes weight reads across $B$ tokens,
raising weight intensity to $I \approx B$ FLOP/byte until $B$ reaches the
ridge point (as measured in that section), though each
sequence still reads its own KV cache (that section).

### Rules of Thumb

the table summarizes the key hardware constants used
throughout the rest of this chapter.

<!-- tbl-caption:Rules of thumb: characteristic magnitudes across the widely deployed GPUs of the table (the newest data-center parts exceed the upper ends), their values on our machine (four RTX 4090s without peer-to-peer access), and why each matters. -->

| quantity | typical magnitude | our machine | why it matters |
|---|---|---|---|
| host time per call, $h$ | 10 µs to 0.1 ms | 13–100 µs (that section) | small operations cannot keep the GPU busy |
| peak bf16 rate, $P$ | 0.17–2.3 PFLOP/s | 165 TFLOP/s | speed limit of large matrix products |
| GPU memory bandwidth, $\beta$ | 1–8 TB/s | ~0.9 TB/s (1.0 rated) | speed limit of elementwise ops and decoding |
| ridge point, $P/\beta$ | 120–300 FLOP/byte | 164 FLOP/byte | below it, memory bandwidth sets the time |
| sweep time, $C/\beta$ | ~20 ms | 24 ms | largest dense model that fits decodes 1 stream at ~40–55 tok/s |
| host to device, pinned | 32–64 GB/s (PCIe 4/5) | ~23 GB/s | keep training data on device or in pinned memory |
| GPU to GPU | 450–900 GB/s (NVLink) | $\le 23$ GB/s via host (that section) | sets the time of every gradient exchange |
| bytes per parameter (Adam) | 16 (fp32), 18 (mixed) | 16–18 | determines what model size fits (that section) |
| bytes per token, decode | $2N$ + KV cache (bf16) | | tokens/s $\lesssim \beta / \textrm{bytes per token}$ |

## Summary

Bandwidth is wire count times rate per wire. Because wires exit a die along
its perimeter while arithmetic units fill its area—and because off-chip data
movement costs hundreds of times more energy than arithmetic—the ridge point
$P/\beta$ lies in the hundreds of FLOP/byte and has risen across hardware
generations. Memory transfers 32-byte sectors from open DRAM rows, so fine-grained
random reads achieve only a fraction of $\beta$, and saturating $\beta$
requires $\beta \times \textrm{latency}$ bytes in flight across warps.
Narrower number formats halve memory traffic and usually double $P$. Finally,
single-stream decoding is bounded by $\beta / (2N + \textrm{KV bytes})$, and
because the memory sweep time $C/\beta$ is ~20 ms across modern GPUs, the
largest dense model that fits on any card decodes a single stream at ~40–55
tokens/s.

## Exercises

1. [code] **Latency and bandwidth, measured.** ● The block transfer model $t(n) = \alpha + n/\beta$ with $n_{1/2} = \alpha\beta$ (that section) describes a single synchronous call.
    1. Predict $n_{1/2}$ for the elementwise multiplication `x * 1.5` of that section, taking $\alpha = h$ and $\beta$ from your measurements in that section.
    1. Time `x * 1.5` for tensor sizes from 1 MB to 2 GB, plot achieved bandwidth against size on log–log axes, and relate the regime transitions to the figure and the on-die L2 cache capacity in the figure.
    1. Repeat the host-to-device measurement of that section for transfers from 4 KB to 256 MB, and fit $\alpha$ and the asymptotic bandwidth.

2. **Wires and rates.** that section writes bandwidth as the number of wires times the rate per wire.
    1. A DDR5-6400 DIMM has 64 data wires at 6.4 Gb/s each. An HBM3e stack has 1,024 data wires, and a B200 carries eight stacks. Compute the bandwidth of one DIMM, the number of DIMMs needed for a 0.5 TB/s CPU socket, and the per-wire data rate implied by the B200's 8 TB/s in the table. Which factor accounts for the gap between the CPU socket and the B200?
    1. Each of the two dies of a B200 sits next to four of the eight HBM stacks. If the SMs on each die read both halves of memory equally often at a combined 8 TB/s, what bandwidth must the on-package bridge carry in each direction?

3. **Starved input pipeline.** Suppose the GPU compute of a training job can
   process 2 GB/s of decoded images. Each CPU worker runs two sequential
   stages: reading compressed JPEGs from NVMe at 1 GB/s (compressed) and
   decoding them on one CPU core at 0.3 GB/s (decoded output, an $8\times$
   size expansion). The training loop then copies each decoded batch synchronously
   from pageable host memory to the GPU at 12 GB/s before computing.
    1. At what rate (in decoded bytes/s) does one worker deliver batches, and
       which stage binds?
    1. How many parallel worker processes are needed to keep the GPU fed?
    1. By what fraction does the synchronous pageable copy lengthen each step,
       and how does switching to asynchronous pinned-memory copies at 24 GB/s
       change this?

4. **Ridge point of a link.** The ridge point $P/\beta$ applies to any data
   path of bandwidth $\beta$, counting the FLOPs a device can execute while
   one byte crosses the path.
    1. Compute $P/\beta$ in bf16 for an RTX 4090 against its own DRAM and
       against the 23 GB/s host-staged GPU-to-GPU path, and for a B200 against
       its HBM and against one direction of NVLink 5 (900 GB/s).
    1. In data-parallel training (that section), each GPU holds a
       model with $N$ parameters, processes $T$ tokens per step at ~$6N$ FLOPs
       per token, and sums bf16 gradients via an allreduce that sends ~$4N$
       bytes per GPU. Without overlap, show that communication takes no longer
       than compute if and only if $T$ exceeds a threshold proportional to the
       link's ridge point, independent of $N$.
    1. Evaluate this token threshold for NVLink 5, for the 23 GB/s pinned-copy
       rate, and for the 2.4 GB/s achieved by default host-staged NCCL on our
       workstation (that section).

5. **Launch-bound decode.** In eager single-stream decoding, the CPU launches
   every kernel of every block while the GPU reads the block's weights. Suppose
   each of $L$ blocks of width $d$ holds $12d^2$ bf16 weights and launches
   $K = 12$ kernels per token at $h = 8$ µs of host time per kernel.
    1. Show that whether host launch time or weight-read time binds is
       independent of $L$, and find the crossover width $d^*$ on the RTX 4090
       and on the B200.
    1. Evaluate both bounds for GPT-2 Small ($d = 768$, $L = 12$) on the
       RTX 4090. Which bound binds?
    1. What does the shift in $d^*$ from the RTX 4090 to the B200 imply for
       the importance of CUDA graph capture-and-replay (that section)
       on faster GPUs?

6. [code] **Block-shuffled gather.** ● Repeat the gather benchmark of
   that section, but permute contiguous blocks of $k$ elements
   for $k \in \{1, 2, 4, \ldots, 1024\}$ instead of individual elements. Plot
   the achieved bandwidth against block size in bytes, and compare the
   crossover size with the 32-byte sector size and the kilobyte scale of a
   DRAM row.

7. **Latency hiding.** ● Apply Little's law (that section) to
   memory latency hiding:
    1. Compute the bytes in flight needed to saturate the B200's 8 TB/s HBM at
       250 ns latency, and compare with the 250 KB of the RTX 4090. What does
       the ~$2\times$ generational growth of bandwidth at nearly constant DRAM
       latency imply for the parallelism kernels must expose?
    1. In pointer chasing, each load's address depends on the previous load,
       allowing only one request in flight per chain. On a CPU with 90 ns DDR5
       latency and 64-byte cache lines, compute the bandwidth of a single
       chain and the number of independent chains required to saturate a
       0.5 TB/s socket.

[Discussions](https://d2l.discourse.group/t/36788)